# ET-aware percentile normalization (per slice)

**Hypothesis.** T1 and T1ce carry the same anatomy at roughly the same intensity distribution,
except that the enhancing tumor adds a high-intensity tail to T1ce. So a single percentile divisor
applied to both is mis-set for T1ce: its 99th percentile sits *inside* the tumor tail, so dividing
by it shrinks T1ce relative to T1 and leaves a residual everywhere, not just at the tumor.

**Proposal.** Normalize FLAIR / T1 / T2 by their 99th percentile over the brain, but normalize T1ce
by the $(99 - 100 f_{ET})$-th percentile, where $f_{ET} = |ET| / |\mathrm{brain}|$ -- the idea being
that the ET voxels are exactly the tail you want to step over.

**Test.** If the hypothesis holds, after normalization $\Delta = \mathrm{T1ce} - \mathrm{T1}$ should
be near 0 across non-tumor brain and clearly positive inside ET.

Four schemes:

| scheme | T1ce divisor | why it is here |
|---|---|---|
| `zscore` | (mean, std) | what `preprocessing/cmap.py` does today -- the baseline |
| `p99` | 99th pct of brain | one percentile for everything -- the thing the proposal fixes |
| `proposed` | $(99 - 100 f_{ET})$-th pct of brain | your rule |
| `oracle` | 99th pct of **brain minus ET** | the quantity `proposed` approximates |

`oracle` is the control that matters: it is what "the 99th percentile of healthy tissue" actually is,
so the gap between it and `proposed` says whether $99 - 100 f_{ET}$ is a good approximation or just
happens to help.

## Everything here is computed PER SLICE

Each slice gets its own divisors and its own $f_{ET}$. Two consequences worth keeping in front of
you:

1. **Most slices contain no ET at all.** On those, $f_{ET} = 0$ and `proposed`, `p99` and `oracle`
   are *identical by construction*. Pooling all slices together therefore dilutes the comparison
   toward "no difference", so every metric below is reported twice: over all brain voxels, and over
   **ET-bearing slices only**. The second is the one that carries information.
2. **Slice-local scaling breaks inter-slice comparability.** A volume normalized per slice is no
   longer consistently scaled along $z$ -- the same tissue can take different values on adjacent
   slices. That is a real change to what the network sees, separate from whether the ET correction
   helps.

`PER_SLICE = False` falls back to the volume-level version, so you can A/B the switch itself.

In [ ]:
import os, glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

# os.chdir('/scratch/ee2178/ImMAP')   # <-- EDIT to your repo root if needed

ROOT = "/home/ee2178/scratch/ee2178/datasets/BraTS/BraTS2021_DataSet_train"   # <-- EDIT
CONTRASTS = ["flair", "t1", "t1ce", "t2"]      # channel order in the h5 (cmap.yaml: contrasts)
FLAIR, T1, T1CE, T2 = 0, 1, 2, 3
PER_SLICE = True      # <-- the mode under test; False = the volume-level version
P_BASE    = 99.0      # base percentile for the "healthy tissue" top
P_FLOOR   = 50.0      # never let the ET correction drag the percentile below this
MIN_VOX   = 200       # slices with fewer brain voxels than this get no reliable percentile
SEED      = 0

subjects = sorted(p for d in sorted(glob.glob(os.path.join(ROOT, "*")))
                  if os.path.isdir(d) for p in glob.glob(os.path.join(d, "*_img.h5")))
if not subjects:
    raise RuntimeError(f"no *_img.h5 under {ROOT}")


def has_keys(p, keys=("img_raw", "mask", "et")):
    with h5py.File(p, "r") as h:
        return all(k in h for k in keys)


usable = [p for p in subjects if has_keys(p)]
print(f"{len(subjects)} subject(s); {len(usable)} with img_raw + mask + et")
print(f"mode: {'PER SLICE' if PER_SLICE else 'per volume'}")
if not usable:
    raise RuntimeError(
        "need 'img_raw' (cmap.yaml save_raw_image: true) and 'et' (save_seg: true). The stored "
        "'img' is already z-scored, so a new normalization cannot be tested on it. Backfill ET "
        "with: python preprocessing/cmap.py --config config/BraTS/cmap.yaml --add-seg-only")
if len(usable) < len(subjects):
    print(f"[warn] {len(subjects) - len(usable)} subject(s) lack img_raw and/or et -- excluded")

## Load one subject

`img_raw` is unnormalized, unclipped, background already zeroed -- so every statistic is taken over
the brain mask, never the whole slice (which is mostly zeros).

In [ ]:
def load_subject(path):
    """-> raw (n,H,W,4) float32 unnormalized, brain (n,H,W) bool, et (n,H,W) bool."""
    with h5py.File(path, "r") as h:
        raw = np.asarray(h["img_raw"]).astype(np.float32)
        brain = np.asarray(h["mask"])[..., 0] > 0.5
        et = np.asarray(h["et"])
        et = (et[..., 0] if et.ndim == 4 else et) > 0.5
    return raw, brain, et


rng = np.random.default_rng(SEED)
SUBJECT = usable[int(rng.integers(len(usable)))]      # <-- or paste a path here to pin one
raw, brain, et = load_subject(SUBJECT)

n = raw.shape[0]
brain_z = brain.reshape(n, -1).sum(1)
et_z = et.reshape(n, -1).sum(1)
f_et_z = et_z / np.maximum(brain_z, 1)
has_et = et_z > 0

print(f"subject : {os.path.basename(os.path.dirname(SUBJECT))}")
print(f"volume  : {n} slices of {raw.shape[1]}x{raw.shape[2]}")
print(f"brain   : {brain.sum():,} voxels    ET: {et.sum():,} "
      f"({et.sum() / max(brain.sum(), 1):.3%} of brain, volume-level)")
print(f"\nper-slice ET fraction over the {int(has_et.sum())}/{n} slices that HAVE ET:")
if has_et.any():
    q = np.percentile(f_et_z[has_et], [0, 25, 50, 75, 100])
    print(f"  min {q[0]:.3%} | p25 {q[1]:.3%} | median {q[2]:.3%} | p75 {q[3]:.3%} | max {q[4]:.3%}")
    print(f"  -> proposed T1ce percentile ranges {P_BASE - 100*q[4]:.2f} .. {P_BASE - 100*q[0]:.2f}")
print(f"{int((~has_et).sum())} slice(s) have NO ET; there all three percentile schemes coincide.")
thin = (brain_z < MIN_VOX) & (brain_z > 0)
if thin.any():
    print(f"[warn] {int(thin.sum())} slice(s) have < {MIN_VOX} brain voxels -- percentiles there "
          f"are noisy (cmap.yaml's drop_empty_slices/min_brain_frac already trims the worst)")

## The four schemes, per slice

In [ ]:
def et_fraction(brain, et, per_slice=PER_SLICE):
    """-> (n,) ET fraction; constant across slices when per_slice is False."""
    n = brain.shape[0]
    if per_slice:
        b = brain.reshape(n, -1).sum(1)
        return et.reshape(n, -1).sum(1) / np.maximum(b, 1)
    return np.full(n, et.sum() / max(brain.sum(), 1), np.float64)


def divisors(vol_c, region, p, per_slice=PER_SLICE):
    """p-th percentile of one contrast over `region`. p is (n,). -> (n,) divisor per slice.
    per_slice=False pools every slice into one percentile and broadcasts it back."""
    n = vol_c.shape[0]
    if not per_slice:
        v = vol_c[region]
        d = float(np.percentile(v, np.clip(float(p[0]), 0.0, 100.0))) if v.size else 1.0
        return np.full(n, d, np.float32)
    out = np.ones(n, np.float32)
    for z in range(n):
        v = vol_c[z][region[z]]
        if v.size:
            out[z] = np.percentile(v, np.clip(float(p[z]), 0.0, 100.0))
    return out


def normalize(raw, brain, et, scheme, p_base=P_BASE, per_slice=PER_SLICE):
    """-> (normalized (n,H,W,4), info). Background stays 0. info['divisor'] is (n,4)."""
    n = raw.shape[0]
    out = np.zeros_like(raw)
    div = np.ones((n, 4), np.float32)
    p_t1ce = np.full(n, p_base, np.float64)

    if scheme == "zscore":                       # what cmap.py does today
        for c in range(4):
            if per_slice:
                for z in range(n):
                    v = raw[z, ..., c][brain[z]]
                    if v.size:
                        div[z, c] = max(float(v.std()), 1e-8)
                        out[z, ..., c] = (raw[z, ..., c] - float(v.mean())) / div[z, c]
            else:
                v = raw[..., c][brain]
                div[:, c] = max(float(v.std()), 1e-8)
                out[..., c] = (raw[..., c] - float(v.mean())) / div[0, c]
        p_t1ce = None
    else:
        if scheme == "proposed":
            p_t1ce = np.maximum(p_base - 100.0 * et_fraction(brain, et, per_slice), P_FLOOR)
        for c in range(4):
            if scheme == "oracle" and c == T1CE:
                div[:, c] = divisors(raw[..., c], brain & ~et, p_t1ce * 0 + p_base, per_slice)
            else:
                div[:, c] = divisors(raw[..., c], brain,
                                     p_t1ce if c == T1CE else np.full(n, p_base), per_slice)
        out = raw / np.maximum(div, 1e-8)[:, None, None, :]
    out = out * brain[..., None]
    return out, {"scheme": scheme, "divisor": div, "pct_t1ce": p_t1ce}


SCHEMES = ["zscore", "p99", "proposed", "oracle"]
norm = {s: normalize(raw, brain, et, s) for s in SCHEMES}

zmax = int(np.argmax(et_z))          # the slice with the most tumor
print(f"divisors on slice {zmax} (the most-ET slice, f_ET = {f_et_z[zmax]:.3%}):\n")
print(f"{'scheme':<10} {'T1 div':>10} {'T1ce div':>10} {'ratio':>8} {'T1ce pct':>10}")
for s in SCHEMES:
    d, p = norm[s][1]["divisor"], norm[s][1]["pct_t1ce"]
    pct = "-" if p is None else f"{p[zmax]:.3f}"
    print(f"{s:<10} {d[zmax, T1]:>10.2f} {d[zmax, T1CE]:>10.2f} "
          f"{d[zmax, T1CE] / max(d[zmax, T1], 1e-8):>8.3f} {pct:>10}")

if has_et.any():
    r = (norm["proposed"][1]["divisor"][has_et, T1CE]
         / np.maximum(norm["oracle"][1]["divisor"][has_et, T1CE], 1e-8) - 1.0)
    print(f"\nproposed vs oracle T1ce divisor, over the {int(has_et.sum())} ET-bearing slices:")
    print(f"  median {np.median(r):+.2%}, IQR [{np.percentile(r, 25):+.2%}, "
          f"{np.percentile(r, 75):+.2%}], |err| < 5% on {np.mean(np.abs(r) < 0.05):.0%} of them")
    print("  <- this is the claim under test: 99 - 100*f_ET standing in for the healthy-tissue p99")

## Divisors along the slice axis

The per-slice view the volume version could not show. On ET-free slices the three percentile schemes
must sit exactly on top of each other; where they separate is where the correction is doing work.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
zz = np.arange(n)

for s in ["p99", "proposed", "oracle"]:
    ax[0].plot(zz, norm[s][1]["divisor"][:, T1CE], lw=1.4, label=s)
ax[0].plot(zz, norm["p99"][1]["divisor"][:, T1], lw=1.0, ls="--", c="k", label="T1 (p99)")
ax[0].set_title("T1ce divisor per slice", fontsize=10)
ax[0].set_xlabel("slice"); ax[0].legend(fontsize=8)

ax[1].plot(zz, norm["proposed"][1]["divisor"][:, T1CE]
           / np.maximum(norm["oracle"][1]["divisor"][:, T1CE], 1e-8) - 1.0, lw=1.4)
ax[1].axhline(0, c="k", lw=0.8)
ax[1].set_title("proposed / oracle - 1   (approximation error)", fontsize=10)
ax[1].set_xlabel("slice"); ax[1].set_ylabel("relative error")

ax[2].plot(zz, 100 * f_et_z, lw=1.4, c="crimson")
ax[2].set_title(r"$100 f_{ET}$ per slice  (= the percentile correction)", fontsize=10)
ax[2].set_xlabel("slice"); ax[2].set_ylabel("percentile points")

for a in ax:
    a.grid(alpha=0.3)
    for z0 in zz[~has_et]:
        a.axvspan(z0 - 0.5, z0 + 0.5, color="0.9", zorder=0)
ax[0].text(0.02, 0.03, "grey = slices with no ET", transform=ax[0].transAxes, fontsize=7)
plt.tight_layout(); plt.show()

## Does $\Delta = $ T1ce $-$ T1 isolate the tumor?

Over non-ET brain we want $\Delta$ **centred on zero** (no global offset) and **tight** (no anatomy
leaking in); inside ET we want it large. `separation` is the ET median over the non-ET RMS.

Reported twice. **all slices** is what the trained network would actually see. **ET slices only**
is the discriminating number -- on ET-free slices the three percentile schemes are identical by
construction, so including them can only pull the schemes together.

In [ ]:
def residual_stats(x, brain, et, keep=None):
    """keep: optional (n,) bool selecting slices to pool over."""
    if keep is not None:
        x, brain, et = x[keep], brain[keep], et[keep]
    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[brain & ~et], d[et]
    if bg.size == 0:
        return {k: float("nan") for k in
                ("bias", "bg_rms", "bg_iqr", "et_med", "separation", "leak")}
    s = {"bias": float(np.median(bg)),                  # want ~0
         "bg_rms": float(np.sqrt(np.mean(bg ** 2))),    # want small
         "bg_iqr": float(np.percentile(bg, 75) - np.percentile(bg, 25)),
         "et_med": float(np.median(tu)) if tu.size else float("nan")}
    s["separation"] = s["et_med"] / max(s["bg_rms"], 1e-8)
    s["leak"] = float(np.mean(bg > 0.5 * s["et_med"])) if tu.size else float("nan")
    return s


for label, keep in [("all slices", None), ("ET slices only", has_et)]:
    rows_ = {s: residual_stats(norm[s][0], brain, et, keep) for s in SCHEMES}
    nsl = n if keep is None else int(keep.sum())
    print(f"--- {label}  ({nsl} slices) ---")
    print(f"{'scheme':<10} {'bias':>9} {'bg_rms':>8} {'bg_iqr':>8} {'et_med':>8} "
          f"{'separation':>11} {'leak':>8}")
    for s in SCHEMES:
        r = rows_[s]
        print(f"{s:<10} {r['bias']:>+9.4f} {r['bg_rms']:>8.4f} {r['bg_iqr']:>8.4f} "
              f"{r['et_med']:>8.4f} {r['separation']:>11.3f} {r['leak']:>8.2%}")
    print()
    if keep is has_et:
        rows = rows_          # keep the discriminating set for the figures below
print("bias -> 0 and separation large = the residual really is just the tumor.")

## Images

The slice with the most ET. Bottom row is $\Delta$ on a symmetric diverging scale (white = 0) with
the ET boundary drawn on -- under the hypothesis it should be flat white outside the contour.
Normalization here is slice-local, so these values are set entirely by this slice's own statistics.

In [ ]:
z = zmax
print(f"slice {z}: ET = {et_z[z]:,} px ({f_et_z[z]:.2%} of this slice's brain)")

ncol = len(SCHEMES) + 1
fig, ax = plt.subplots(2, ncol, figsize=(3.0 * ncol, 6.2))
ax[0][0].imshow(raw[z, ..., T1] * brain[z], cmap="gray")
ax[0][0].set_title("T1 (raw)", fontsize=9)
ax[1][0].imshow(raw[z, ..., T1CE] * brain[z], cmap="gray")
ax[1][0].set_title("T1ce (raw)", fontsize=9)
for a in (ax[0][0], ax[1][0]):
    a.contour(et[z], levels=[0.5], colors="r", linewidths=0.6)

for j, s in enumerate(SCHEMES, start=1):
    x = norm[s][0]
    lo, hi = np.percentile(x[z, ..., T1CE][brain[z]], [1, 99])
    ax[0][j].imshow(x[z, ..., T1CE], cmap="gray", vmin=lo, vmax=hi)
    ax[0][j].set_title(f"{s}\nT1ce normalized", fontsize=9)
    d = (x[z, ..., T1CE] - x[z, ..., T1]) * brain[z]
    v = float(np.percentile(np.abs(d[brain[z]]), 99)) or 1.0
    im = ax[1][j].imshow(d, cmap="bwr", vmin=-v, vmax=v)
    ax[1][j].contour(et[z], levels=[0.5], colors="k", linewidths=0.6)
    sep_z = residual_stats(x[z:z + 1], brain[z:z + 1], et[z:z + 1])["separation"]
    ax[1][j].set_title(f"$\\Delta$   sep(this slice)={sep_z:.2f}", fontsize=9)
    plt.colorbar(im, ax=ax[1][j], fraction=0.046)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Histograms

Pooled over **ET-bearing slices only**, for the reason above. Left: T1 vs T1ce over the brain -- the
hypothesis says these should overlap once T1ce's tumor tail is stepped over. Right: the residual,
split healthy / ET. Under the hypothesis the healthy curve is a narrow spike at 0 and the ET curve
sits clearly to its right.

In [ ]:
sel = has_et if has_et.any() else np.ones(n, bool)
fig, ax = plt.subplots(2, len(SCHEMES), figsize=(3.3 * len(SCHEMES), 6.0))
for j, s in enumerate(SCHEMES):
    x, b_, e_ = norm[s][0][sel], brain[sel], et[sel]
    t1, t1ce = x[..., T1][b_], x[..., T1CE][b_]
    lo, hi = np.percentile(np.concatenate([t1, t1ce]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 120)
    ax[0][j].hist(t1, bins=bins, histtype="step", density=True, label="T1")
    ax[0][j].hist(t1ce, bins=bins, histtype="step", density=True, label="T1ce")
    ax[0][j].set_title(s, fontsize=10); ax[0][j].set_yscale("log")
    if j == 0:
        ax[0][j].legend(fontsize=8); ax[0][j].set_ylabel("brain, log density")

    d = x[..., T1CE] - x[..., T1]
    bg, tu = d[b_ & ~e_], d[e_]
    b2 = np.linspace(*np.percentile(d[b_], [0.5, 99.5]), 120)
    ax[1][j].hist(bg, bins=b2, histtype="step", density=True, label="healthy")
    if tu.size:
        ax[1][j].hist(tu, bins=b2, histtype="step", density=True, label="ET")
    ax[1][j].axvline(0, color="k", lw=0.8); ax[1][j].set_yscale("log")
    ax[1][j].set_xlabel(r"$\Delta$ = T1ce - T1")
    if j == 0:
        ax[1][j].legend(fontsize=8); ax[1][j].set_ylabel("log density")
plt.tight_layout(); plt.show()

## Robustness across subjects

One subject proves nothing -- per-slice $f_{ET}$ varies enormously, and so does how well
$99 - 100 f_{ET}$ tracks the oracle. This resamples `N_SUBJECTS` at random. All statistics are
pooled over **ET-bearing slices** of each subject.

In [ ]:
N_SUBJECTS = 20

rng = np.random.default_rng(SEED)
pick = rng.choice(len(usable), size=min(N_SUBJECTS, len(usable)), replace=False)

recs, slice_err = [], []
for i, si in enumerate(pick):
    p = usable[int(si)]
    try:
        r_, b_, e_ = load_subject(p)
    except Exception as err:
        print(f"[skip] {os.path.basename(p)}: {type(err).__name__}")
        continue
    nz = r_.shape[0]
    ez = e_.reshape(nz, -1).sum(1)
    keep = ez > 0
    if not keep.any():
        print(f"[skip] {os.path.basename(os.path.dirname(p))}: no ET voxels")
        continue
    rec = {"subject": os.path.basename(os.path.dirname(p)),
           "n_et_slices": int(keep.sum()), "n_slices": nz,
           "f_et": float(np.median((ez / np.maximum(b_.reshape(nz, -1).sum(1), 1))[keep])),
           "div": {}}
    for s in SCHEMES:
        x_, i_ = normalize(r_, b_, e_, s)
        rec[s] = residual_stats(x_, b_, e_, keep)
        rec["div"][s] = i_["divisor"][keep, T1CE]
    slice_err.append(rec["div"]["proposed"] / np.maximum(rec["div"]["oracle"], 1e-8) - 1.0)
    recs.append(rec)
    print(f"  {i + 1}/{len(pick)} {rec['subject']}  "
          f"{rec['n_et_slices']}/{nz} ET slices, median f_ET={rec['f_et']:.2%}      ", end="\r")

print(f"\n\n{len(recs)} subject(s) with ET; "
      f"{sum(r['n_et_slices'] for r in recs)} ET-bearing slices in total\n")
print(f"{'scheme':<10} {'|bias| med (IQR)':>24} {'bg_rms med (IQR)':>24} "
      f"{'separation med (IQR)':>26} {'leak med':>10}")
for s in SCHEMES:
    q = lambda k: np.nanpercentile([abs(r[s][k]) if k == "bias" else r[s][k] for r in recs],
                                   [25, 50, 75])
    b, g, sep, lk = q("bias"), q("bg_rms"), q("separation"), q("leak")
    print(f"{s:<10} {b[1]:>10.4f} ({b[0]:.3f}-{b[2]:.3f}) {g[1]:>10.4f} ({g[0]:.3f}-{g[2]:.3f}) "
          f"{sep[1]:>12.2f} ({sep[0]:.2f}-{sep[2]:.2f}) {lk[1]:>9.2%}")

allerr = np.concatenate(slice_err)
print(f"\nproposed / oracle T1ce divisor, PER SLICE over {allerr.size} ET-bearing slices:")
print(f"  median {np.median(allerr):+.2%}, IQR [{np.percentile(allerr, 25):+.2%}, "
      f"{np.percentile(allerr, 75):+.2%}], |err| < 5% on {np.mean(np.abs(allerr) < 0.05):.0%}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
xs = np.arange(len(SCHEMES))
for a, key, ttl in zip(ax, ["separation", "bg_rms", "bias"],
                       ["separation (higher = tumor stands out)",
                        "non-ET RMS (lower = flatter background)",
                        "non-ET bias (closer to 0 = no global offset)"]):
    for k, s in enumerate(SCHEMES):
        v = [r[s][key] for r in recs]
        a.scatter(np.full(len(v), k) + rng.normal(0, 0.06, len(v)), v, s=12, alpha=0.6)
        a.scatter([k], [np.nanmedian(v)], marker="_", s=600, c="k", zorder=3)
    a.set_xticks(xs); a.set_xticklabels(SCHEMES, rotation=20)
    a.set_title(ttl, fontsize=9); a.grid(alpha=0.3)
    if key == "bias":
        a.axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter([r["f_et"] for r in recs],
              [r["proposed"]["separation"] - r["p99"]["separation"] for r in recs], s=18)
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xlabel(r"median per-slice $f_{ET}$"); ax[0].set_ylabel("separation gain\n(proposed - p99)")
ax[0].set_title("the correction should help most where the tail is biggest", fontsize=9)

ax[1].hist(allerr, bins=60)
ax[1].axvline(0, color="k", lw=0.8)
ax[1].set_xlabel("proposed / oracle - 1, per ET-bearing slice")
ax[1].set_title("does the rule land on the healthy-tissue percentile?", fontsize=9)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Caveats

* **Slice-local scaling breaks comparability along $z$.** Each slice is divided by its own
  percentile, so the same tissue can take different values on neighbouring slices. Whether that is
  acceptable is a separate question from whether the ET correction works -- flip `PER_SLICE` to
  compare. Per-slice will always look flatter on the residual metrics (more free parameters), so
  judge the *switch* on downstream training, not on `bg_rms` here.
* **Per-slice $f_{ET}$ is far noisier than the volume value**, and on a slice clipping the edge of
  the tumor a handful of ET voxels sets the correction. `MIN_VOX` flags thin slices but does not
  exclude them.
* **The rule assumes the ET voxels are exactly the top $f_{ET}$ of the distribution.** They are not:
  some enhancement is dimmer than the brightest healthy tissue (vessels, fat), and necrotic core
  inside the tumor is dark. `oracle` is immune to this and `proposed` is not, which is why the
  proposed/oracle divisor error is the honest measure -- more so than the separation numbers, which
  can improve for the wrong reason.
* **ET is one label.** `cmap.yaml` sets `et_labels: [4]`; whole-tumor or edema would give a very
  different $f_{ET}$.
* **This needs a mask at normalization time, which you do not have at inference.** If the scheme
  works, the next question is a mask-free stand-in -- a fixed percentile offset fit to these results,
  or estimating the tail fraction from the T1ce/T1 histogram divergence directly.